In [ ]:
# analysis.ipynb

# Импорты
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.dataset import MovieDataset, get_dataloader
from src.model import NeuralCF
import torch
from sklearn.metrics import mean_squared_error
import numpy as np

# Пути к файлам (используем исправленные ASCII версии)
RATINGS_PATH = '../data/ratings_ascii.dat'
MOVIES_PATH = '../data/movies_ascii.dat'
USERS_PATH = '../data/users_ascii.dat'  # если нужно для других функций

# Получаем DataLoader и размеры
dataloader, n_users, n_movies, user_encoder, movie_encoder = get_dataloader(
    RATINGS_PATH, MOVIES_PATH, batch_size=64
)

# Загружаем модель
model = NeuralCF(n_users, n_movies)
model.load_state_dict(torch.load('../neural_cf_model.pth'))
model.eval()

# EDA: Распределение рейтингов
dataset = MovieDataset(RATINGS_PATH, MOVIES_PATH)
ratings_df = dataset.ratings

plt.figure(figsize=(8,5))
sns.histplot(ratings_df['rating'], bins=10, kde=False)
plt.title("Распределение рейтингов")
plt.xlabel("Рейтинг")
plt.ylabel("Количество оценок")
plt.show()

# Оценка модели
y_true = []
y_pred = []

with torch.no_grad():
    for user, movie, rating in dataloader:
        output = model(user, movie)
        y_true.extend(rating.numpy())
        y_pred.extend(output.numpy())

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE модели Neural CF: {rmse:.4f}")
